## Litter Detection Subsystem Computer Vision Training

The following code is intended to be run via a cloud compute program such as Kaggle or Google Colab. It downloads the TACO dataset, splits it into train/test/val sets, and fine-tunes a selected ultralytics model on the data.

The initial training of this dataset had a number of images less specific to grass removed manually via the file structure, which may provide more similar performance to Model_1_engine_2.engine, which was the model used for testing. 

In [1]:
# KAGGLE-SPECIFIC STARTUP

# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!git clone https://github.com/pedropro/TACO

Cloning into 'TACO'...
remote: Enumerating objects: 740, done.
remote: Counting objects: 100% (435/435), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 740 (delta 416), reused 380 (delta 380), pack-reused 305 (from 1)
Receiving objects: 100% (740/740), 97.48 MiB | 27.23 MiB/s, done.
Resolving deltas: 100% (499/499), done.


In [3]:
%cd TACO
!pip3 install -r requirements.txt
!pip3 install pycocotools


/kaggle/working/TACO


In [4]:
# Download Dataset
%cd TACO

import json
import os
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

ANNOTATIONS = "data/annotations.json"
DATA_ROOT   = Path("data")
MAX_WORKERS = 16  # parallel downloads

with open(ANNOTATIONS) as f:
    coco = json.load(f)

def download_image(img):
    url       = img["flickr_url"]
    file_name = Path(img["file_name"])
    dest      = DATA_ROOT / file_name
    dest.parent.mkdir(parents=True, exist_ok=True)

    if dest.exists():
        return f"Skipped (exists): {file_name}"

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        with open(dest, "wb") as f:
            f.write(response.content)
        return f"Downloaded: {file_name}"
    except Exception as e:
        # Try coco_url as fallback only if it's a real URL
        coco_url = img.get("coco_url")
        if not coco_url:
            return f"Failed: {file_name} — {e} (no fallback URL)"
        try:
            response = requests.get(coco_url, timeout=10)
            response.raise_for_status()
            with open(dest, "wb") as f:
                f.write(response.content)
            return f"Downloaded (fallback): {file_name}"
        except Exception as e2:
            return f"Failed: {file_name} — {e2}"

images = coco["images"]
print(f"Downloading {len(images)} images with {MAX_WORKERS} workers...")

failed = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(download_image, img): img for img in images}
    for i, future in enumerate(as_completed(futures)):
        result = future.result()
        if "Failed" in result:
            failed.append(result)
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(images)} complete")

print(f"\nDone. {len(images) - len(failed)} succeeded, {len(failed)} failed.")
if failed:
    for f in failed:
        print(f"  {f}")

[Errno 2] No such file or directory: 'TACO'
/kaggle/working/TACO
  50/1500 complete
  100/1500 complete
  150/1500 complete
  200/1500 complete
  250/1500 complete
  300/1500 complete
  350/1500 complete
  400/1500 complete
  450/1500 complete
  500/1500 complete
  550/1500 complete
  600/1500 complete
  650/1500 complete
  700/1500 complete
  750/1500 complete
  800/1500 complete
  850/1500 complete
  900/1500 complete
  950/1500 complete
  1000/1500 complete
  1050/1500 complete
  1100/1500 complete
  1150/1500 complete
  1200/1500 complete
  1250/1500 complete
  1300/1500 complete
  1350/1500 complete
  1400/1500 complete
  1450/1500 complete
  1500/1500 complete

Done. 1500 succeeded, 0 failed.


In [5]:
os.chdir('/kaggle/working/TACO')
import json
import shutil
from pathlib import Path
from collections import defaultdict
from sklearn.model_selection import train_test_split

# ── Config ────────────────────────────────────────────────────────────────────
ANNOTATIONS  = "data/annotations.json"
DATA_ROOT    = Path("data")
OUTPUT_ROOT  = Path("dataset")
TRAIN_RATIO  = 0.7
VAL_RATIO    = 0.15
TEST_RATIO   = 0.15
RANDOM_SEED  = 42

# ── Load annotations ──────────────────────────────────────────────────────────
with open(ANNOTATIONS) as f:
    coco = json.load(f)

image_info = {img["id"]: img for img in coco["images"]}

anns_by_image = defaultdict(list)
for ann in coco["annotations"]:
    anns_by_image[ann["image_id"]].append(ann)

# ── Convert segmentation polygons to YOLO format ──────────────────────────────
def get_yolo_lines(image_id):
    img = image_info[image_id]
    w, h = img["width"], img["height"]
    lines = []
    for ann in anns_by_image[image_id]:
        for polygon in ann["segmentation"]:
            if isinstance(polygon, dict) or len(polygon) < 6:
                continue
            coords = []
            for i in range(0, len(polygon), 2):
                coords.append(f"{polygon[i]     / w:.6f}")
                coords.append(f"{polygon[i + 1] / h:.6f}")
            lines.append("0 " + " ".join(coords))  # always class 0
    return lines

# ── Split image IDs into train / val / test ───────────────────────────────────
all_ids = [i for i in image_info.keys() if anns_by_image[i]]

train_ids, tmp_ids = train_test_split(all_ids, test_size=(VAL_RATIO + TEST_RATIO), random_state=RANDOM_SEED)
val_ids,  test_ids = train_test_split(tmp_ids,  test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO), random_state=RANDOM_SEED)

print(f"Train: {len(train_ids)}  Val: {len(val_ids)}  Test: {len(test_ids)}")

# ── Write images + labels into split folders ──────────────────────────────────
def write_split(ids, split_name):
    img_out = OUTPUT_ROOT / split_name / "images"
    lbl_out = OUTPUT_ROOT / split_name / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    skipped = 0
    for image_id in ids:
        img      = image_info[image_id]
        src_path = DATA_ROOT / img["file_name"]

        if not src_path.exists():
            skipped += 1
            continue

        flat_name = img["file_name"].replace("/", "_").replace("\\", "_")
        shutil.copy(src_path, img_out / flat_name)

        lines = get_yolo_lines(image_id)
        label_name = Path(flat_name).stem + ".txt"
        with open(lbl_out / label_name, "w") as f:
            f.write("\n".join(lines))

    if skipped:
        print(f"  [{split_name}] Skipped {skipped} missing images")

write_split(train_ids, "train")
write_split(val_ids,   "val")
write_split(test_ids,  "test")

# ── Write taco.yaml ───────────────────────────────────────────────────────────
yaml_content = """\
path: dataset
train: train/images
val:   val/images
test:  test/images

nc: 1
names: ["litter"]
"""

with open("taco.yaml", "w") as f:
    f.write(yaml_content)

print("Done. YAML written to taco.yaml")

Train: 1050  Val: 225  Test: 225
Done. YAML written to taco.yaml


In [6]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.9 MB/s eta 0:00:00


In [ ]:
# ENSURE GPUs SELECTED, NOT JUST CPU

import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  VRAM total:     {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")
    print(f"  VRAM available: {torch.cuda.mem_get_info(i)[0] / 1024**3:.1f} GB")

# Confirm device list is valid
device_count = torch.cuda.device_count()
device = list(range(device_count))  
print(f"Using devices: {device}")

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
  VRAM total:     14.6 GB
  VRAM available: 14.5 GB
GPU 1: Tesla T4
  VRAM total:     14.6 GB
  VRAM available: 14.5 GB
Using devices: [0, 1]


In [8]:
from ultralytics import YOLO

model = YOLO("yolo26n-seg.pt") # Choose desired model

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
import os
os.chdir("/kaggle/working/TACO")

model.train(
    data="taco.yaml",
    epochs=150,
    imgsz=1280,
    batch=12,
    optimizer="auto",
    cos_lr=True,
    multi_scale=False,
    overlap_mask=True,
    device=list(range(torch.cuda.device_count())),
    dropout=0.1,
    warmup_epochs=5,
    patience=15,
    mixup=0.1,
    copy_paste=0.3,
    flipud=0.2,
    degrees=45.0,
    scale=0.75,
    cache=True,
    plots=True,
    project="/kaggle/working/runs",
    name="taco_train",
    save=True,
    save_period=10,
) # Training parameters may be altered for greater performance, see issue of the final report for the originally decided parameters. 

Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
                                                      CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=taco.yaml, degrees=45.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=taco_train, nbs=64, nms=False, opse

In [14]:
# Export the trained weights as an ONNX file, 
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/taco_train/weights/best.pt")
model.export(
    format="onnx",
    imgsz=1280,
    half=True,
    simplify=True,
) 

# The model can be then downloaded via the file strucuture of Colab, Kaggle, etc. It is expected the model should be converted to a Jetson Orin Nano specific TensorRT .engine file for greatly increased performance.

Ultralytics 8.4.26 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26s-seg summary (fused): 139 layers, 10,365,727 parameters, 0 gradients, 34.1 GFLOPs

PyTorch: starting from '/kaggle/working/runs/taco_train/weights/best.pt' with input shape (1, 3, 1280, 1280) BCHW and output shape(s) ((1, 300, 38), (1, 32, 320, 320)) (22.4 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 302ms
Prepared 2 packages in 2.77s
Installed 2 packages in 19ms
 + onnxruntime-gpu==1.24.4
 + onnxslim==0.1.90

requirements: AutoUpdate success ✅ 4.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 20...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:5353: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  warnings.warn(


ONNX: slimming with onnxslim 0.1.90...
ONNX: converting to FP16...
ONNX: export success ✅ 9.8s, saved as '/kaggle/working/runs/taco_train/weights/best.onnx' (20.2 MB)

Export complete (14.1s)
Results saved to /kaggle/working/runs/taco_train/weights
Predict:         yolo predict task=segment model=/kaggle/working/runs/taco_train/weights/best.onnx imgsz=1280 half
Validate:        yolo val task=segment model=/kaggle/working/runs/taco_train/weights/best.onnx imgsz=1280 data=taco.yaml half 
Visualize:       https://netron.app


'/kaggle/working/runs/taco_train/weights/best.onnx'

In [ ]:
# Output dataset as a ZIP, not necessary if just the model is required. 
import shutil

shutil.make_archive(
    "/kaggle/working/taco_dataset",   # output zip name
    "zip",
    "/kaggle/working/TACO/dataset",                # directory to zip
)

In [ ]:
from IPython.display import FileLink
FileLink(r'taco_dataset.zip')